# Local Simulated Annealing

This notebook contains the full N-Queens implementation for **Local Simulated Annealing**.

It does all of the following when you run the main code cell:

- solves the required cases `N = 10, 30, 50, 100, 200, 500`
- prints a structured console table
- saves results in CSV format only
- saves a board SVG visual for the small solved case
- saves a performance graph image
- updates the combined comparison CSV and comparison chart


In [ ]:
"""
Project Title: Solving the N-Queens Problem with Exhaustive Search and Genetic Algorithms

File: simulated_annealing_nqueens.py
Approach: Local Search using Simulated Annealing

Design Notes
- This script uses a permutation board, so each column is used exactly once.
- The annealing move swaps two rows and evaluates diagonal conflicts only.
- A light local-repair pass is used at the end if annealing stalls very close to a solution.
- Results are printed in a structured table and saved automatically after execution.
"""

from __future__ import annotations

import csv
import math
import random
import time
import tracemalloc
from dataclasses import dataclass
from html import escape
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown, SVG


PROJECT_TITLE = "Solving the N-Queens Problem with Exhaustive Search and Genetic Algorithms"
ALGORITHM_NAME = "Local Simulated Annealing"
FILE_STEM = "simulated_annealing_nqueens"
TEST_SIZES = [10, 30, 50, 100, 200, 500]
BOARD_VISUAL_LIMIT = 12
DEFAULT_SEED = 42


@dataclass
class RunResult:
    n: int
    status: str
    solved: bool
    steps: int
    conflicts: int | str
    time_seconds: float
    memory_mb: float
    note: str
    solution: list[int] | None = None
    image_path: str = "N/A"
    image_path: str = "N/A"


def pair_conflicts(counts) -> int:
    total = 0
    for count in counts:
        if count > 1:
            total += count * (count - 1) // 2
    return total


def count_conflicts(board: list[int]) -> int:
    n = len(board)
    diag_main = [0] * (2 * n - 1)
    diag_anti = [0] * (2 * n - 1)

    for row, col in enumerate(board):
        diag_main[row - col + n - 1] += 1
        diag_anti[row + col] += 1

    return pair_conflicts(diag_main) + pair_conflicts(diag_anti)


def render_board(board: list[int] | None) -> str:
    if board is None:
        return "No solution board available."

    n = len(board)
    if n > BOARD_VISUAL_LIMIT:
        return f"Board visual skipped for N = {n}. Small-board rendering is limited to N <= {BOARD_VISUAL_LIMIT}."

    lines = ["Solution Vector: [" + ", ".join(str(value) for value in board) + "]"]
    header = "    " + " ".join(f"{col:>2}" for col in range(n))
    border = "   +" + "---" * n + "+"
    lines.append(header)
    lines.append(border)

    for row, queen_col in enumerate(board):
        row_cells = ["Q" if col == queen_col else "." for col in range(n)]
        lines.append(f"{row:>2} | " + "  ".join(row_cells) + " |")

    lines.append(border)
    return "\n".join(lines)


def save_board_svg(board: list[int], title: str, output_path: Path) -> Path:
    n = len(board)
    cell_size = 56
    margin_left = 82
    margin_top = 110
    footer_height = 88
    board_size = n * cell_size
    width = margin_left + board_size + 40
    height = margin_top + board_size + footer_height
    colors = {
        "light": "#edf6f9",
        "dark": "#83c5be",
        "queen": "#006d77",
        "outline": "#264653",
        "title": "#0b132b",
        "subtitle": "#3a506b",
        "label": "#1c2541",
    }
    solution_vector = "Solution vector: [" + ", ".join(str(value) for value in board) + "]"

    svg_parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_left}" y="42" font-size="28" font-family="Segoe UI, Arial, sans-serif" font-weight="700" fill="{colors["title"]}">{escape(title)}</text>',
        f'<text x="{margin_left}" y="74" font-size="20" font-family="Consolas, Courier New, monospace" fill="{colors["subtitle"]}">{escape(solution_vector)}</text>',
    ]

    for col in range(n):
        x = margin_left + col * cell_size + cell_size / 2
        svg_parts.append(
            f'<text x="{x}" y="{margin_top - 20}" text-anchor="middle" font-size="18" font-family="Segoe UI, Arial, sans-serif" fill="{colors["label"]}">{col}</text>'
        )

    for row in range(n):
        y = margin_top + row * cell_size + cell_size / 2 + 7
        svg_parts.append(
            f'<text x="{margin_left - 28}" y="{y}" text-anchor="middle" font-size="18" font-family="Segoe UI, Arial, sans-serif" fill="{colors["label"]}">{row}</text>'
        )
        for col in range(n):
            x = margin_left + col * cell_size
            y_cell = margin_top + row * cell_size
            fill = colors["light"] if (row + col) % 2 == 0 else colors["dark"]
            svg_parts.append(f'<rect x="{x}" y="{y_cell}" width="{cell_size}" height="{cell_size}" fill="{fill}" stroke="{colors["outline"]}" stroke-width="1.2"/>')
            if board[row] == col:
                cx = x + cell_size / 2
                cy = y_cell + cell_size / 2
                svg_parts.append(f'<circle cx="{cx}" cy="{cy}" r="{cell_size * 0.26}" fill="{colors["queen"]}" stroke="#ffffff" stroke-width="2.5"/>')
                svg_parts.append(f'<text x="{cx}" y="{cy + 8}" text-anchor="middle" font-size="28" font-family="Segoe UI Symbol, Arial Unicode MS, serif" fill="#ffffff">Q</text>')

    svg_parts.append(f'<text x="{margin_left}" y="{height - 28}" font-size="18" font-family="Segoe UI, Arial, sans-serif" fill="{colors["subtitle"]}">Simulated annealing board snapshot saved for notebook and report visuals.</text>')
    svg_parts.append("</svg>")
    output_path.write_text("\n".join(svg_parts), encoding="utf-8")
    return output_path


def save_board_svg(board: list[int], title: str, output_path: Path) -> Path:
    n = len(board)
    cell_size = 56
    margin_left = 82
    margin_top = 110
    footer_height = 88
    board_size = n * cell_size
    width = margin_left + board_size + 40
    height = margin_top + board_size + footer_height
    colors = {
        "light": "#83c5be",
        "dark": "#006d77",
        "queen": "#ffffff",
        "outline": "#006d77",
        "title": "#006d77",
        "subtitle": "#006d77",
        "label": "#006d77",
    }
    solution_vector = "Solution vector: [" + ", ".join(str(value) for value in board) + "]"

    svg_parts = [
        f'<svg xmlns="http://www.w3.org/2000/svg" width="{width}" height="{height}" viewBox="0 0 {width} {height}">',
        '<rect width="100%" height="100%" fill="#ffffff"/>',
        f'<text x="{margin_left}" y="42" font-size="28" font-family="Segoe UI, Arial, sans-serif" font-weight="700" fill="{colors["title"]}">{escape(title)}</text>',
        f'<text x="{margin_left}" y="74" font-size="20" font-family="Consolas, Courier New, monospace" fill="{colors["subtitle"]}">{escape(solution_vector)}</text>',
    ]

    for col in range(n):
        x = margin_left + col * cell_size + cell_size / 2
        svg_parts.append(
            f'<text x="{x}" y="{margin_top - 20}" text-anchor="middle" font-size="18" font-family="Segoe UI, Arial, sans-serif" fill="{colors["label"]}">{col}</text>'
        )

    for row in range(n):
        y = margin_top + row * cell_size + cell_size / 2 + 7
        svg_parts.append(
            f'<text x="{margin_left - 28}" y="{y}" text-anchor="middle" font-size="18" font-family="Segoe UI, Arial, sans-serif" fill="{colors["label"]}">{row}</text>'
        )
        for col in range(n):
            x = margin_left + col * cell_size
            y_cell = margin_top + row * cell_size
            fill = colors["light"] if (row + col) % 2 == 0 else colors["dark"]
            svg_parts.append(f'<rect x="{x}" y="{y_cell}" width="{cell_size}" height="{cell_size}" fill="{fill}" stroke="{colors["outline"]}" stroke-width="1.2"/>')
            if board[row] == col:
                cx = x + cell_size / 2
                cy = y_cell + cell_size / 2
                svg_parts.append(f'<circle cx="{cx}" cy="{cy}" r="{cell_size * 0.26}" fill="{colors["outline"]}" stroke="#ffffff" stroke-width="2.4"/>')
                svg_parts.append(f'<text x="{cx}" y="{cy + 8}" text-anchor="middle" font-size="28" font-family="Segoe UI Symbol, Arial Unicode MS, serif" fill="{colors["queen"]}">Q</text>')

    svg_parts.append(f'<text x="{margin_left}" y="{height - 28}" font-size="18" font-family="Segoe UI, Arial, sans-serif" fill="{colors["subtitle"]}">Board visual saved for notebook presentation and report use.</text>')
    svg_parts.append("</svg>")
    output_path.write_text("\n".join(svg_parts), encoding="utf-8")
    return output_path


def save_performance_chart(results: list[RunResult], output_path: Path) -> Path:
    plt.style.use("seaborn-v0_8-whitegrid")
    ns = [item.n for item in results]
    times = [item.time_seconds for item in results]
    memories = [item.memory_mb for item in results]
    statuses = [item.status for item in results]
    colors = ["#006d77" if status == "SOLVED" else "#f59e0b" if status == "TIMEOUT" else "#dc2626" for status in statuses]

    fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
    axes[0].plot(ns, times, color="#006d77", linewidth=2.4, marker="o", markersize=7)
    axes[0].scatter(ns, times, c=colors, s=90, edgecolors="#ffffff", linewidths=1.2, zorder=3)
    axes[0].set_title(f"{ALGORITHM_NAME} - Execution Time", fontsize=13, fontweight="bold")
    axes[0].set_xlabel("Board Size (N)")
    axes[0].set_ylabel("Time (seconds)")
    axes[0].set_xticks(ns)
    axes[0].tick_params(axis="x", rotation=20)

    axes[1].bar([str(value) for value in ns], memories, color="#83c5be", edgecolor="#006d77", linewidth=1.2)
    axes[1].set_title(f"{ALGORITHM_NAME} - Peak Memory", fontsize=13, fontweight="bold")
    axes[1].set_xlabel("Board Size (N)")
    axes[1].set_ylabel("Peak Memory (MB)")

    for axis in axes:
        axis.grid(True, alpha=0.25)

    fig.suptitle(PROJECT_TITLE, fontsize=15, fontweight="bold")
    fig.tight_layout()
    fig.savefig(output_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return output_path


def refresh_combined_outputs(output_dir: Path) -> tuple[Path, Path]:
    comparison_csv = output_dir / "all_algorithms_comparison.csv"
    comparison_chart = output_dir / "charts" / "all_algorithms_comparison.png"
    source_files = [
        ("Exhaustive Depth-First Search", output_dir / "dfs_nqueens_results.csv"),
        ("Local Greedy Search (Hill Climbing)", output_dir / "greedy_nqueens_results.csv"),
        ("Local Simulated Annealing", output_dir / "simulated_annealing_nqueens_results.csv"),
        ("Genetic Algorithm", output_dir / "genetic_nqueens_results.csv"),
    ]

    combined_rows: list[dict[str, str]] = []
    for algorithm, csv_path in source_files:
        if not csv_path.exists():
            continue
        with csv_path.open("r", newline="", encoding="utf-8") as csv_file:
            reader = csv.DictReader(csv_file)
            for row in reader:
                step_value = row.get("Steps") or row.get("Iterations") or row.get("Generations") or ""
                combined_rows.append(
                    {
                        "Algorithm": algorithm,
                        "N": row.get("N", ""),
                        "Status": row.get("Status", ""),
                        "Solved": row.get("Solved", ""),
                        "Steps": step_value,
                        "Conflicts": row.get("Conflicts", ""),
                        "TimeSeconds": row.get("TimeSeconds", ""),
                        "PeakMemoryMB": row.get("PeakMemoryMB", ""),
                        "ImagePath": row.get("ImagePath", "N/A"),
                    }
                )

    combined_rows.sort(key=lambda item: (item["Algorithm"], int(item["N"])))

    with comparison_csv.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(
            csv_file,
            fieldnames=["Algorithm", "N", "Status", "Solved", "Steps", "Conflicts", "TimeSeconds", "PeakMemoryMB", "ImagePath"],
        )
        writer.writeheader()
        writer.writerows(combined_rows)

    if combined_rows:
        plt.style.use("seaborn-v0_8-whitegrid")
        fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
        algorithms = [item[0] for item in source_files]
        palette = {
            "Exhaustive Depth-First Search": "#c0392b",
            "Local Greedy Search (Hill Climbing)": "#2d6a4f",
            "Local Simulated Annealing": "#006d77",
            "Genetic Algorithm": "#9d0208",
        }

        for algorithm in algorithms:
            rows = [row for row in combined_rows if row["Algorithm"] == algorithm]
            if not rows:
                continue
            ns = [int(row["N"]) for row in rows]
            times = [float(row["TimeSeconds"]) for row in rows]
            memories = [float(row["PeakMemoryMB"]) for row in rows]
            axes[0].plot(ns, times, marker="o", linewidth=2.2, label=algorithm, color=palette[algorithm])
            axes[1].plot(ns, memories, marker="o", linewidth=2.2, label=algorithm, color=palette[algorithm])

        axes[0].set_title("Execution Time Comparison", fontsize=13, fontweight="bold")
        axes[0].set_xlabel("Board Size (N)")
        axes[0].set_ylabel("Time (seconds)")
        axes[0].set_xticks([10, 30, 50, 100, 200, 500])
        axes[0].tick_params(axis="x", rotation=20)

        axes[1].set_title("Peak Memory Comparison", fontsize=13, fontweight="bold")
        axes[1].set_xlabel("Board Size (N)")
        axes[1].set_ylabel("Peak Memory (MB)")
        axes[1].set_xticks([10, 30, 50, 100, 200, 500])
        axes[1].tick_params(axis="x", rotation=20)

        for axis in axes:
            axis.grid(True, alpha=0.25)
            axis.legend(fontsize=8)

        fig.suptitle(PROJECT_TITLE, fontsize=15, fontweight="bold")
        fig.tight_layout()
        fig.savefig(comparison_chart, dpi=180, bbox_inches="tight")
        plt.close(fig)

    return comparison_csv, comparison_chart


def format_table(headers: list[str], rows: list[list[str]]) -> str:
    widths = [len(header) for header in headers]
    for row in rows:
        for index, value in enumerate(row):
            widths[index] = max(widths[index], len(value))

    border = "+" + "+".join("-" * (width + 2) for width in widths) + "+"

    def render_row(values: list[str]) -> str:
        return "| " + " | ".join(value.ljust(widths[index]) for index, value in enumerate(values)) + " |"

    lines = [border, render_row(headers), border]
    for row in rows:
        lines.append(render_row(row))
    lines.append(border)
    return "\n".join(lines)


def conflicted_rows(board: list[int]) -> list[int]:
    n = len(board)
    rows: list[int] = []
    for row, col in enumerate(board):
        clashes = 0
        for other_row, other_col in enumerate(board):
            if other_row == row:
                continue
            if abs(other_col - col) == abs(other_row - row):
                clashes += 1
        if clashes:
            rows.append(row)
    return rows


def local_repair(board: list[int], rng: random.Random, max_steps: int) -> tuple[list[int], bool, int]:
    candidate = board[:]
    n = len(candidate)

    for step in range(1, max_steps + 1):
        current_conflicts = count_conflicts(candidate)
        if current_conflicts == 0:
            return candidate, True, step - 1

        rows = conflicted_rows(candidate)
        if not rows:
            return candidate, True, step - 1

        row = rng.choice(rows)
        best_score = None
        best_swap_row = row

        for other_row in range(n):
            if other_row == row:
                continue
            candidate[row], candidate[other_row] = candidate[other_row], candidate[row]
            score = count_conflicts(candidate)
            candidate[row], candidate[other_row] = candidate[other_row], candidate[row]

            if best_score is None or score < best_score:
                best_score = score
                best_swap_row = other_row

        candidate[row], candidate[best_swap_row] = candidate[best_swap_row], candidate[row]

    return candidate, count_conflicts(candidate) == 0, max_steps


def solve_simulated_annealing(n: int, seed: int = DEFAULT_SEED) -> tuple[list[int] | None, bool, int, str]:
    if n in (2, 3):
        return None, False, 0, "No valid solution exists for N = 2 or N = 3."

    rng = random.Random(seed)
    max_restarts = 4
    max_steps = max(5000, 300 * n)
    cooling_rate = 0.9995
    initial_temperature = max(25.0, n * 0.4)
    total_steps = 0
    best_board: list[int] | None = None
    best_conflicts = float("inf")

    for restart in range(max_restarts):
        board = list(range(n))
        rng.shuffle(board)
        temperature = initial_temperature
        current_conflicts = count_conflicts(board)

        if current_conflicts < best_conflicts:
            best_conflicts = current_conflicts
            best_board = board[:]

        for _ in range(max_steps):
            total_steps += 1
            if current_conflicts == 0:
                return board[:], True, total_steps, f"Solved with simulated annealing after {restart} restart(s)."

            first_row, second_row = rng.sample(range(n), 2)
            board[first_row], board[second_row] = board[second_row], board[first_row]
            next_conflicts = count_conflicts(board)
            delta = next_conflicts - current_conflicts

            if delta <= 0 or rng.random() < math.exp(-delta / max(temperature, 1e-9)):
                current_conflicts = next_conflicts
                if current_conflicts < best_conflicts:
                    best_conflicts = current_conflicts
                    best_board = board[:]
            else:
                board[first_row], board[second_row] = board[second_row], board[first_row]

            temperature *= cooling_rate

    if best_board is not None:
        repaired_board, repaired, repair_steps = local_repair(best_board, rng, max_steps=max(200, 10 * n))
        if repaired:
            total_steps += repair_steps
            return repaired_board, True, total_steps, "Solved after annealing plus final local repair."

    final_note = f"Annealing stopped after {max_restarts} restart(s); best board had {int(best_conflicts)} conflict(s)."
    return best_board, False, total_steps, final_note


def run_case(n: int) -> RunResult:
    tracemalloc.start()
    start_time = time.perf_counter()
    solution, solved, steps, note = solve_simulated_annealing(n)
    elapsed = time.perf_counter() - start_time
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    conflicts = count_conflicts(solution) if solution is not None else "N/A"
    status = "SOLVED" if solved else "FAILED"

    return RunResult(
        n=n,
        status=status,
        solved=solved,
        steps=steps,
        conflicts=conflicts,
        time_seconds=elapsed,
        memory_mb=peak / (1024 * 1024),
        note=note,
        solution=solution,
    )




def save_outputs(results: list[RunResult]) -> tuple[Path, Path, Path, Path, list[Path]]:
    output_dir = Path("outputs")
    charts_dir = output_dir / "charts"
    image_dir = output_dir / "board_images"
    output_dir.mkdir(exist_ok=True)
    charts_dir.mkdir(exist_ok=True)
    image_dir.mkdir(exist_ok=True)

    csv_path = output_dir / f"{FILE_STEM}_results.csv"
    image_index_path = output_dir / f"{FILE_STEM}_images.csv"
    chart_path = charts_dir / f"{FILE_STEM}_performance.png"

    saved_images: list[Path] = []
    for item in results:
        if item.solution is not None and item.n <= BOARD_VISUAL_LIMIT:
            image_path = image_dir / f"{FILE_STEM}_n{item.n}.svg"
            save_board_svg(item.solution, f"{ALGORITHM_NAME} - N = {item.n}", image_path)
            item.image_path = str(image_path)
            saved_images.append(image_path)

    with csv_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["N", "Status", "Solved", "Iterations", "Conflicts", "TimeSeconds", "PeakMemoryMB", "ImagePath", "Note"])
        for item in results:
            writer.writerow([
                item.n,
                item.status,
                item.solved,
                item.steps,
                item.conflicts,
                f"{item.time_seconds:.6f}",
                f"{item.memory_mb:.6f}",
                item.image_path,
                item.note,
            ])

    with image_index_path.open("w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(["N", "ImagePath"])
        for item in results:
            if item.image_path != "N/A":
                writer.writerow([item.n, item.image_path])

    save_performance_chart(results, chart_path)
    combined_csv_path, combined_chart_path = refresh_combined_outputs(output_dir)
    return csv_path, chart_path, combined_csv_path, combined_chart_path, saved_images


def main() -> None:
    print(PROJECT_TITLE)
    print(f"Algorithm: {ALGORITHM_NAME}")
    print(f"Testing board sizes: {TEST_SIZES}")
    print()

    results = [run_case(n) for n in TEST_SIZES]
    rows = [
        [
            str(item.n),
            item.status,
            str(item.steps),
            str(item.conflicts),
            f"{item.time_seconds:.6f}",
            f"{item.memory_mb:.6f}",
            item.note,
        ]
        for item in results
    ]
    print(format_table(["N", "Status", "Iterations", "Conflicts", "Time (sec)", "Memory (MB)", "Note"], rows))
    print()

    first_small_solution = next((item for item in results if item.solution is not None and item.n <= BOARD_VISUAL_LIMIT), None)
    if first_small_solution is not None:
        print(f"Board Visual for N = {first_small_solution.n}")
        print(render_board(first_small_solution.solution))
        print()


    csv_path, chart_path, combined_csv_path, combined_chart_path, saved_images = save_outputs(results)
    if saved_images:
        display(Markdown("### Saved Board Visual"))
        display(SVG(filename=str(saved_images[0])))
    display(Markdown("### Saved Performance Chart"))
    display(Image(filename=str(chart_path)))
    print(f"Saved CSV results to: {csv_path}")
    print(f"Saved performance chart to: {chart_path}")
    print(f"Saved combined comparison CSV to: {combined_csv_path}")
    print(f"Saved combined comparison chart to: {combined_chart_path}")


if __name__ == "__main__":
    main()
